In [1]:
#Dynamic search path configuration
import sys
import os

#Adjusting search path based on notebook's depth in directory --> iterates through directories until it finds directory name: US_conflict_forecasting
current_dir = os.getcwd()
while 'US_conflict_forecasting' not in os.path.basename(current_dir):
    current_dir = os.path.dirname(current_dir)

#Adjusting search path based on 'new' current directory
if current_dir not in sys.path:
    sys.path.insert(0, current_dir)

from project_setup import setup_project
project_root_dir = setup_project()

Current Project Root Directory set to: /Users/luca/Documents/GitHub/US_conflict_forecasting
Current Search Path set to: ['/Users/luca/Documents/GitHub/US_conflict_forecasting', '/Users/luca/Documents/GitHub/US_conflict_forecasting/1_Features/b_Conflict_Features/Conflict_MergeClean', '/Users/luca/anaconda3/envs/ML_Lab/lib/python312.zip', '/Users/luca/anaconda3/envs/ML_Lab/lib/python3.12', '/Users/luca/anaconda3/envs/ML_Lab/lib/python3.12/lib-dynload', '', '/Users/luca/anaconda3/envs/ML_Lab/lib/python3.12/site-packages', '/Users/luca/Documents/GitHub/US_conflict_forecasting']


In [2]:
import numpy as np
import pandas as pd

In [3]:
#Loading data 
ACLED_df = pd.read_csv(os.path.join(project_root_dir, '2_Dataset/b_Partial_Datasets/joined_ACLED_ZIP_22_PA.csv'))

#Actor Features 
ACLED_Actors = pd.read_csv(os.path.join(project_root_dir, '2_Dataset/b_Partial_Datasets/ACLED_ZIP_Actor_22.csv'))

#Subevent Features 
ACLED_Subevents = pd.read_csv(os.path.join(project_root_dir, '2_Dataset/b_Partial_Datasets/ACLED_ZIP_Subevent_22.csv'))

#Topic Features
ACLED_Topics = pd.read_csv(os.path.join(project_root_dir, '2_Dataset/b_Partial_Datasets/ACLED_ZIP_Topic.csv'))

#### Restructuring Date Column

In [4]:
#Extracting year and month from date column

#Parsing date column to datetime
ACLED_df['parsed_date'] = pd.to_datetime(ACLED_df['event_date'], format='%d %B %Y')

#Extracting year and month
ACLED_df['event_MY'] = ACLED_df['parsed_date'].dt.to_period('M')

#### Joining Partial Conflict Feature Datasets

In [5]:
ACLED_merged = ACLED_df.merge(ACLED_Actors, on=['event_id_cnty', 'ZCTA5CE10'], how='left') \
              .merge(ACLED_Subevents, on=['event_id_cnty', 'ZCTA5CE10'], how='left') \
              .merge(ACLED_Topics, on=['event_id_cnty', 'ZCTA5CE10'], how='left')

#Dropping duplicate cells
ACLED_merged = ACLED_merged.loc[:,~ACLED_merged.columns.duplicated()]

#### Aggregation function 
Can add more if needed

In [6]:
#Defining aggregation across features
aggregation_function = {
    'admin2': 'first', 
    'fatalities': 'sum',  
    'actor_type1': 'max',
    'actor_type2': 'max',
    'actor_type3': 'max',
    'actor_type4': 'max',
    'actor_type5': 'max',
    'actor_type6': 'max',
    'actor_type7': 'max',
    'actor_type8': 'max',
    'actor_type9': 'max', 
    'set_peaceful_prot': 'max', 
    'set_other': 'max', 
    'set_prot_int': 'max', 
    'set_mob': 'max', 
    'set_violent_prot': 'max', 
    'set_loot_destr': 'max', 
    'unions': 'max', 
    'environment': 'max',
    'gun-violence': 'max',
    'healthcare': 'max', 
}

#### Grouping by ZIP Code and Month

In [7]:
#Grouping by ZIP Code and Month
#agg aggregates over the previously specified parameters
grouped = ACLED_merged.groupby(['ZCTA5CE10', 'event_MY']).agg(aggregation_function).reset_index()

#Adding event count per ZIP Code and Month
grouped['event_count'] = ACLED_merged.groupby(['ZCTA5CE10', 'event_MY']).size().reset_index(name='event_count')['event_count']

print(grouped.head(10))

   ZCTA5CE10 event_MY        admin2  fatalities  actor_type1  actor_type2  \
0      15001  2022-01        Beaver           0            0            0   
1      15001  2022-09        Beaver           0            1            1   
2      15009  2022-06        Beaver           0            1            1   
3      15010  2022-05        Beaver           0            0            0   
4      15025  2022-05     Allegheny           0            0            0   
5      15026  2022-10     Allegheny           0            0            0   
6      15026  2022-12     Allegheny           0            0            0   
7      15037  2022-07     Allegheny           0            0            0   
8      15068  2022-03  Westmoreland           0            0            0   
9      15068  2022-07  Westmoreland           0            0            1   

   actor_type3  actor_type4  actor_type5  actor_type6  ...  set_other  \
0            0            0            0            0  ...          0   
1     

In [8]:
#Creating an index 'date range' of all months (in order to allocate each ZIP code to each month) 
date_range = pd.period_range(start='2022-01', end='2022-12', freq='M')
print(date_range)

PeriodIndex(['2022-01', '2022-02', '2022-03', '2022-04', '2022-05', '2022-06',
             '2022-07', '2022-08', '2022-09', '2022-10', '2022-11', '2022-12'],
            dtype='period[M]')


In [9]:
#Dataframe mapping ZIP Codes to Counties 
zip_county = ACLED_merged[['ZCTA5CE10', 'admin2']].drop_duplicates()

In [10]:
#New dataframe combining date range and ZIP Code
#Using multiindex object to combine unique ZIP code list with date range index 
all_combinations = pd.MultiIndex.from_product([ACLED_merged['ZCTA5CE10'].unique(), date_range], names=['ZCTA5CE10', 'event_MY'])

In [11]:
#Initializing df with all combinations 
#Resetting index for default index 
restructured_df = pd.DataFrame(index=all_combinations).reset_index()

In [12]:
#Merge 1
#Merging zip_county df with initialized df for all combinations
restructured_df = restructured_df.merge(zip_county, on='ZCTA5CE10', how='left')

In [13]:
#Merge 2 
#Merging initialized df with grouped data holding aggregates and counts
restructured_df = restructured_df.merge(grouped, on=['ZCTA5CE10', 'admin2', 'event_MY'], how='left')

In [14]:
#Filling missing values + converting to int
restructured_df['event_count'] = restructured_df['event_count'].fillna(0).astype(int)
restructured_df['fatalities'] = restructured_df['fatalities'].fillna(0).astype(int)
restructured_df['actor_type1'] = restructured_df['actor_type1'].fillna(0).astype(int)
restructured_df['actor_type2'] = restructured_df['actor_type2'].fillna(0).astype(int)
restructured_df['actor_type3'] = restructured_df['actor_type3'].fillna(0).astype(int)
restructured_df['actor_type4'] = restructured_df['actor_type4'].fillna(0).astype(int)
restructured_df['actor_type5'] = restructured_df['actor_type5'].fillna(0).astype(int)
restructured_df['actor_type6'] = restructured_df['actor_type6'].fillna(0).astype(int)
restructured_df['actor_type7'] = restructured_df['actor_type7'].fillna(0).astype(int)
restructured_df['actor_type8'] = restructured_df['actor_type8'].fillna(0).astype(int)
restructured_df['actor_type9'] = restructured_df['actor_type9'].fillna(0).astype(int)
restructured_df['set_peaceful_prot'] = restructured_df['set_peaceful_prot'].fillna(0).astype(int)
restructured_df['set_other'] = restructured_df['set_other'].fillna(0).astype(int)
restructured_df['set_prot_int'] = restructured_df['set_prot_int'].fillna(0).astype(int)
restructured_df['set_mob'] = restructured_df['set_mob'].fillna(0).astype(int)
restructured_df['set_violent_prot'] = restructured_df['set_violent_prot'].fillna(0).astype(int)
restructured_df['set_loot_destr'] = restructured_df['set_loot_destr'].fillna(0).astype(int)
restructured_df['unions'] = restructured_df['unions'].fillna(0).astype(int)
restructured_df['environment'] = restructured_df['environment'].fillna(0).astype(int)
restructured_df['gun-violence'] = restructured_df['gun-violence'].fillna(0).astype(int)
restructured_df['healthcare'] = restructured_df['healthcare'].fillna(0).astype(int)



In [15]:
#Sorting df by ZIP and time
restructured_df = restructured_df.sort_values(by=['ZCTA5CE10', 'event_MY'])
print(restructured_df.head(50))

      ZCTA5CE10 event_MY     admin2  fatalities  actor_type1  actor_type2  \
564       15001  2022-01     Beaver           0            0            0   
565       15001  2022-02     Beaver           0            0            0   
566       15001  2022-03     Beaver           0            0            0   
567       15001  2022-04     Beaver           0            0            0   
568       15001  2022-05     Beaver           0            0            0   
569       15001  2022-06     Beaver           0            0            0   
570       15001  2022-07     Beaver           0            0            0   
571       15001  2022-08     Beaver           0            0            0   
572       15001  2022-09     Beaver           0            1            1   
573       15001  2022-10     Beaver           0            0            0   
574       15001  2022-11     Beaver           0            0            0   
575       15001  2022-12     Beaver           0            0            0   

In [16]:
#Renaming columns
restructured_df.rename(columns={'ZCTA5CE10': 'ZCTA', 
                                'actor_type1': 'act_gov',  
                                'actor_type2': 'act_pol',
                                'actor_type3': 'act_nat',
                                'actor_type4': 'act_rel',
                                'actor_type5': 'act_edu',
                                'actor_type6': 'act_wor',
                                'actor_type7': 'act_iss',
                                'actor_type8': 'act_rwm',
                                'actor_type9': 'act_oth',}, inplace=True)

In [17]:
#checking unique ZIPS 
unique_ZCTAs = restructured_df['ZCTA'].nunique()

# Display the unique values
print(unique_ZCTAs)

124


In [18]:
#Exporting to csv
output_directory_1 = os.path.join(project_root_dir, '2_Dataset/b_Partial_Datasets')
output_filename_1 = "ConflictFeaturesClean.csv"
output_path_1 = os.path.join(output_directory_1, output_filename_1)

restructured_df.to_csv(output_path_1, index=False)